In [ ]:
from transformers import CLIPProcessor, CLIPModel

In [ ]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
# dir(clip_model)

In [ ]:
clip_model.num_parameters()

In [ ]:
def get_text_embedding(text):
    inputs = clip_processor(text=text, return_tensors="pt", padding=True)
    text_features = clip_model.get_text_features(**inputs)
    return text_features

In [ ]:
output = get_text_embedding("A photo of a cat")
output.pooler_output.shape

In [ ]:
output.last_hidden_state.shape

In [ ]:
import requests
from PIL import Image

In [ ]:
def get_image_embedding(image_url):
    response = requests.get(image_url, stream=True, headers={"User-Agent": "Vivek (vivek@vivek.com)"}).raw
    image = Image.open(response)
    inputs = clip_processor(images=image, return_tensors="pt", padding=True)
    image_features = clip_model.get_image_features(**inputs)
    return image_features

    

In [ ]:
image_url = "https://cdn.overstockart.com/blog/wp-content/uploads/2018/04/GS2840-1020x500.jpg"
image_features = get_image_embedding(image_url)

In [ ]:
image_features.pooler_output.shape

In [ ]:
from IPython.display import Image as DisplayImage

In [ ]:
DisplayImage(image_url)

In [ ]:
text = "a picnic spot by the side of the river"
text_features = get_text_embedding(text)

In [ ]:
import torch

In [ ]:
torch.nn.functional.cosine_similarity(text_features.pooler_output, image_features.pooler_output)

In [ ]:
text = "kids with their parents"
text_features = get_text_embedding(text)
torch.nn.functional.cosine_similarity(text_features.pooler_output, image_features.pooler_output)


In [ ]:
text = "motorcycle"
text_features = get_text_embedding(text)
torch.nn.functional.cosine_similarity(text_features.pooler_output, image_features.pooler_output)


In [ ]:
text = "big buildings"
text_features = get_text_embedding(text)
torch.nn.functional.cosine_similarity(text_features.pooler_output, image_features.pooler_output)


In [ ]:
text = "kids with their parents"
text_features = get_text_embedding(text)
torch.nn.functional.cosine_similarity(text_features.pooler_output, image_features.pooler_output)


In [ ]:
import seaborn as sns

In [ ]:
sns.histplot(text_features.pooler_output.tolist())

In [ ]:
print(text_features.pooler_output.mean())
print(text_features.pooler_output.std())
print(text_features.pooler_output.max())
print(text_features.pooler_output.min())
print(text_features.pooler_output.median())
print(text_features.pooler_output.quantile(0.95))


In [ ]:
(text_features.pooler_output**2).sum().sqrt()

In [ ]:
from torchvision.utils import make_grid
from torchvision.io import read_image

In [ ]:
def get_text_embeddings(text):
    inputs = clip_processor(text=text, return_tensors="pt")
    text_embeddings = clip_model.get_text_features(**inputs)
    return text_embeddings.pooler_output


def get_image_embeddings(image_paths):
    # clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    # clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    images = [read_image(image_path) for image_path in image_paths]
    inputs = clip_processor(images=images, return_tensors="pt")
    image_embeddings = clip_model.get_image_features(**inputs).pooler_output
    return image_embeddings


In [ ]:
from pathlib import Path

In [ ]:
images = []
images.extend(Path("../../data/images").glob("*.jpg"))

In [ ]:
len(images)

In [ ]:
images[-2:]

In [ ]:
images[0]
display(Image.open(images[196]))

In [ ]:
image_embeddings = get_image_embeddings(images[-2:])
# image_embeddings.pooler_output.shape
# image_embeddings.last_hidden_state.shape
image_embeddings.shape

In [ ]:
import numpy as np

In [ ]:
batch_size = 20
total_batches = int(np.ceil(len(images) / batch_size))
print(total_batches)

In [ ]:
batch_size = 20
total_batches = int(np.ceil(len(images) / batch_size))
embedding_files = []

for i in range(0, len(images), batch_size):
    batch = images[i : i + batch_size]
    print(f"Processing batch {i//batch_size+1}/{total_batches}")
    file_name = f"../../data/image_embeddings/batch_{i//batch_size+1}.pt"
    torch.save(get_image_embeddings(batch), file_name)
    embedding_files.append(file_name)

In [ ]:
image_embeddings = torch.cat([torch.load(file) for file in embedding_files])

In [ ]:
image_embeddings.shape

In [ ]:
from tqdm import tqdm

In [ ]:
from IPython.display import Image as DisplayImage, display

In [ ]:
from torchvision import transforms

In [ ]:
def rank_images(text_embedding, image_embeddings):
    scores = []
    cosine_similarity = torch.nn.functional.cosine_similarity
    for image_embedding in tqdm(image_embeddings):
        similarities = cosine_similarity(text_embedding, image_embedding, dim=1)
        scores.append(float(similarities))
    return scores

In [ ]:
text_embedding = get_text_embeddings("water bottle")
similarities = rank_images(text_embedding, image_embeddings)

In [ ]:
# similarities

In [ ]:
# np.array(images)[idx]

In [ ]:
idx = np.argsort(similarities)[-10:][::-1]
i = 0
for image in np.array(images)[idx]:
    display(DisplayImage(image, width=150, height=150))
    i += 1
    if i >= 5:
        break

In [ ]:
from torchvision.transforms.functional import rotate

In [ ]:
target_size = (480, 720)
num_images = 6
resized_transform = transforms.Resize(target_size)
resized_images = [resized_transform(rotate(read_image(image), -90)) for image in np.array(images)[idx][:num_images]]
grid_image = make_grid(resized_images, nrow=2)
pil_image = transforms.ToPILImage()(grid_image)
pil_image.resize((800, 600))
display(pil_image)

In [ ]:
def rank_images(text_embedding, image_embeddings):
    cosine_similarity = torch.nn.functional.cosine_similarity
    similarities = cosine_similarity(text_embedding, image_embeddings)
    return similarities

In [ ]:
similarities = rank_images(get_text_embeddings("a lunch box"), image_embeddings)
idx = np.argsort(similarities.detach().numpy())[-10:][::-1]
idx

In [ ]:
def generate_relevant_images(query, image_embeddings, num_images=6):
    text_embedding = get_text_embeddings(query)
    similarities = rank_images(text_embedding, image_embeddings)
    idx = np.argsort(similarities.detach().numpy())[-num_images:][::-1]
    ranked_images = np.array(images)[idx]

    target_size = (240, 480)
    resized_transform = transforms.Resize(target_size)
    resized_images = [resized_transform(rotate(read_image(image), -90)) for image in ranked_images]
    grid_image = make_grid(resized_images, nrow=2)
    pil_image = transforms.ToPILImage()(grid_image)
    # pil_image.resize((800, 600))
    display(pil_image)
    return ranked_images



In [ ]:
generate_relevant_images("price tag", image_embeddings) 